# Module 2: Dataset Exploration and Understanding
**Project:** An Interpretable Multimodal Framework for Cardiotoxicity Risk Assessment  
**Dataset:** `BC_cardiotox` (Breast Cancer Cardiotoxicity Dataset)  
**Reference:** Piñeiro-Lamas et al., *Scientific Data* (2023), Nature Publishing Group. [DOI: 10.1038/s41597-023-02419-1](https://doi.org/10.1038/s41597-023-02419-1)

---
### Objectives
1. Catalog the dataset directory inventory and verify file integrity.
2. Inspect the clinical, functional, and combined CSV files.
3. Characterize the CTRCD target label and analyze class imbalance.
4. Profile missing values across all variables.
5. Investigate patient identifiers, duplicate records, and multimodal mapping.
6. Inspect the Tissue Doppler Imaging (TDI) representations.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Set paths relative to workspace root
BASE_DIR = Path("..")
DATASET_DIR = BASE_DIR / "dataset"
IMAGES_DIR = DATASET_DIR / "IMAGES"

print(f"Dataset Directory: {DATASET_DIR.resolve()}")

## 1. Load the Dataset CSVs
The dataset uses European formatting: semicolon (`;`) delimiter and comma (`,`) decimal separator.

In [ ]:
df_clin = pd.read_csv(DATASET_DIR / "BC_cardiotox_clinical_variables.csv", sep=';', decimal=',')
df_func = pd.read_csv(DATASET_DIR / "BC_cardiotox_functional_variable.csv", sep=';', decimal=',')
df_comb = pd.read_csv(DATASET_DIR / "BC_cardiotox_clinical_and_functional_variables.csv", sep=';', decimal=',')

print(f"Clinical Data Shape:   {df_clin.shape} (531 patients, 27 features)")
print(f"Functional Data Shape: {df_func.shape} (270 patients, 1 target + 1001 time points)")
print(f"Combined Data Shape:   {df_comb.shape} (531 patients, 27 clinical + 1001 functional)")

## 2. CTRCD Target Label & Class Imbalance
The primary prediction target is `CTRCD` (Cancer Therapy-Related Cardiac Dysfunction):
- `0`: Negative (No cardiotoxicity developed)
- `1`: Positive (Cardiotoxicity observed during follow-up)

In [ ]:
func_mask = ~df_comb.iloc[:, 27].isna()

summary_data = []
for name, subset in [
    ("Clinical Dataset", df_clin['CTRCD']),
    ("Functional Dataset", df_func['CTRCD']),
    ("Combined Overall", df_comb['CTRCD']),
    ("Combined w/ TDI", df_comb.loc[func_mask, 'CTRCD']),
    ("Combined w/o TDI", df_comb.loc[~func_mask, 'CTRCD'])
]:
    vc = subset.value_counts()
    neg, pos = int(vc.get(0, 0)), int(vc.get(1, 0))
    total = neg + pos
    summary_data.append({
        "Subset": name,
        "Total": total,
        "CTRCD=0 (Neg)": neg,
        "CTRCD=1 (Pos)": pos,
        "Prevalence (%)": round(pos / total * 100, 2),
        "Ratio": f"{round(neg / pos, 2)}:1"
    })

pd.DataFrame(summary_data)

## 3. Missing Value Profiling
Profiling missing data across clinical features and functional columns.

In [ ]:
null_counts = df_clin.isnull().sum()
null_cols = null_counts[null_counts > 0].sort_values(ascending=False)
null_df = pd.DataFrame({
    "Missing Count": null_cols,
    "Missing (%)": (null_cols / len(df_clin) * 100).round(2)
})
print(f"Total missing cells in clinical dataset: {null_counts.sum()} / {df_clin.size} ({(null_counts.sum() / df_clin.size * 100):.2f}%)")
print(f"Clinical columns with zero missing values: {list(null_counts[null_counts == 0].index)}")
null_df.head(15)

## 4. Multimodal Row Alignment & Permutation Discovery
We investigate whether the 270 functional records match the first 270 rows of the combined dataset, and how the combined dataset maps to the clinical CSV.

In [ ]:
# Verify that the first 270 rows of combined functional columns match df_func exactly
comb_func = df_comb.iloc[:270, 27:].values
func_vals = df_func.iloc[:, 1:].values
diff = np.max(np.abs(comb_func - func_vals))
print(f"Max absolute difference between combined rows 0..269 and df_func: {diff:.6e} (Exact match!)")

# Check target sequence alignment
target_match = (df_comb.iloc[:270]['CTRCD'].values == df_func['CTRCD'].values).all()
print(f"Do CTRCD labels match row-for-row between combined rows 0..269 and df_func? {target_match}")

## 5. TDI Numerical Waveforms (Discretized Doppler Curves)
Plotting the mean myocardial velocity curve for CTRCD negative vs positive patients.

In [ ]:
time_grid = np.linspace(0, 1, 1001)
targets = df_func['CTRCD'].values
waveforms = df_func.iloc[:, 1:].values

mean_neg = waveforms[targets == 0].mean(axis=0)
std_neg = waveforms[targets == 0].std(axis=0)
mean_pos = waveforms[targets == 1].mean(axis=0)
std_pos = waveforms[targets == 1].std(axis=0)

plt.figure(figsize=(10, 4.5))
plt.fill_between(time_grid, mean_neg - std_neg, mean_neg + std_neg, alpha=0.15, color='#2b5c8f')
plt.fill_between(time_grid, mean_pos - std_pos, mean_pos + std_pos, alpha=0.2, color='#e0564c')
plt.plot(time_grid, mean_neg, label=f'CTRCD=0 Mean (N={(targets==0).sum()})', color='#2b5c8f', lw=2)
plt.plot(time_grid, mean_pos, label=f'CTRCD=1 Mean (N={(targets==1).sum()})', color='#e0564c', lw=2, linestyle='--')
plt.axhline(0, color='gray', linestyle=':', lw=0.8)
plt.xlabel('Normalized Cardiac Cycle [0, 1]')
plt.ylabel('Velocity (cm/s)')
plt.title('TDI Myocardial Velocity Waveforms (Mean ± 1 SD)')
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

## 6. TDI Image Inspection
Inspecting sample images from `dataset/IMAGES/`.

In [ ]:
sample_path = IMAGES_DIR / "image (1).png"
with Image.open(sample_path) as im:
    print(f"Sample Image: {sample_path.name}")
    print(f"Dimensions:   {im.size[0]} x {im.size[1]}")
    print(f"Color Mode:   {im.mode}")
    plt.figure(figsize=(8, 4.5))
    plt.imshow(im)
    plt.title(f"{sample_path.name}: Spectral Doppler Velocity Display (957x717)")
    plt.axis('off')
    plt.show()